# Data Indexing

In [7]:
import os
from dotenv import load_dotenv

from langchain.document_loaders import PyPDFLoader, CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore import InMemoryDocstore
import faiss

In [8]:
DATA_PATH = "../data"

In [9]:
EMBEDDINGS = OpenAIEmbeddings()
INDEX = faiss.IndexFlatL2(len(OpenAIEmbeddings().embed_query(" ")))
DOCSTORE = InMemoryDocstore({ })
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

In [10]:
vectorstore = FAISS(
    embedding_function = EMBEDDINGS,
    index = INDEX,
    docstore = DOCSTORE,
    index_to_docstore_id = {}
)

In [11]:
for file_name in os.listdir(DATA_PATH):
    file_name = os.path.join(DATA_PATH, file_name)

    if file_name.endswith(".pdf"):
        loader = PyPDFLoader(file_path = file_name)
        docs = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size = CHUNK_SIZE,
            chunk_overlap = CHUNK_OVERLAP,
            length_function = len
        )
        text = text_splitter.split_documents(docs)

        vectorstore.add_documents(text)

    elif file_name.endswith(".csv"):
        loader = CSVLoader(file_path = file_name)
        docs = loader.load_and_split()
        
        vectorstore.add_documents(documents = docs)

    else:
        print(f"Found unsupported file format: {file_name}")

Found unsupported file format: ../data\faiss_index


In [12]:
vectorstore.save_local(os.path.join(DATA_PATH, "faiss_index"))